# 🏪 Zava Agentic Fine-Tuning Lab — 03: Baseline & Grader

**In this notebook**, you'll:
1. Evaluate base o4-mini on our validation scenarios to establish a **baseline**
2. Understand how the **grader** works — the key to RFT learning
3. Calibrate the **pass threshold** for optimal training signal

| What you'll do | Time |
|----------------|------|
| Define the grader function | 3 min |
| Run baseline evaluation on 8 scenarios | 7 min |
| Understand RFT vs SFT | 2 min |
| Calibrate pass threshold | 3 min |

> **Prerequisite**: Complete `01-introduction-setup.ipynb` first.

---
## Setup — Reconnect and Load Agent Infrastructure

We need the full agent pipeline (client, tools, runner) to evaluate the model.

In [ ]:
import json, os, re, time, textwrap
import requests
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential = DefaultAzureCredential()
)
client = project_client.get_openai_client(api_key=os.environ["AZURE_OPENAI_API_KEY"])
print("✅ Connected to Microsoft Foundry")

✅ Connected to Microsoft Foundry


In [14]:
# === Tool endpoint (pre-deployed Azure Function) ===
TOOL_URL = "https://zava-rft-tools.azurewebsites.net"

# The system prompt the agent uses
SYSTEM_PROMPT = """You are Zava's return resolution engine. Call get_order to look up order details, then apply the return policy to compute the resolution.

POLICY: Standard=30d/15d(electronics), Gold=45d/30d, Platinum=60d/45d. Electronics restocking: Std=15%, Gold=7.5%, Plat=0%. Defective=0%. Sale=final sale (defective sale→store credit). Late delivery(>2d)=$10 credit +15d extension. Lost=replacement/refund. Pending=cancellable. Opened personal care=deny unless defective.

Respond with your resolution including: action, amounts, and policy reasoning."""

# Tool definition (same schema the model sees)
TOOLS = [
    {"type": "function", "function": {
        "name": "get_order",
        "description": "Look up order details including items, prices, dates, loyalty tier, and delivery status.",
        "parameters": {"type": "object", "properties": {
            "order_id": {"type": "string", "description": "The order ID (e.g., ORD-003)"}
        }, "required": ["order_id"]}
    }}
]


def call_tool(name, args):
    """Call the Zava tool endpoint and return the result."""
    url = f"{TOOL_URL}/tool/{name}"
    payload = {"arguments": json.dumps(args), "call_id": "c", "id": "f", "trace_id": "t"}
    r = requests.post(url, json=payload, timeout=30)
    return r.json().get("output", json.dumps(r.json()))


def run_agent(user_message, model="o4-mini", verbose=True):
    """Run the full agent loop: model → tool call → model → response."""
    messages = [
        {"role": "developer", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    tool_calls_made = []

    for turn in range(8):  # max 8 turns to prevent infinite loops
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, max_completion_tokens=8192
        )
        msg = resp.choices[0].message

        # Build assistant message for conversation history
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
            tool_calls_made.extend(msg.tool_calls)
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            if verbose and msg.content:
                print(f"\n📋 Agent Response:\n{textwrap.fill(msg.content, width=80)}")
            return msg.content or "", tool_calls_made

        # Execute tool calls
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"  🔧 Calling {tc.function.name}({args})")
            result = call_tool(tc.function.name, args)
            if verbose:
                # Show a preview of the tool result
                preview = result[:200] + "..." if len(result) > 200 else result
                print(f"  📦 Result: {preview}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "", tool_calls_made


---
## Section 3: Baseline Evaluation (10 min)

Before fine-tuning, we need to know how well the base model performs.
We'll score it on validation scenarios using a **grader** that checks:
- Did it get the right **action** (refund, deny, store credit)? (40%)
- Did it compute the correct **dollar amounts**? (30%)
- Did it cite the right **policy reasons**? (20%)
- Did it **use the tool**? (10%)

### Define the Grader

This is the **same grading function** used during RFT training. Understanding it
is key to understanding how the model learns.

In [15]:
def python_grader(output_text, output_tools, expected_resolution):
    """Score a model response against the expected resolution.
    Returns 0.0 to 1.0 — same logic used during RFT training."""
    if not expected_resolution:
        return 0.5

    score = 0.0
    exp_lower = expected_resolution.lower()
    out_lower = (output_text or "").lower()

    # Action correctness (0.4)
    actions = {
        "refund": ["refund"],
        "denied": ["denied", "deny", "not eligible", "cannot", "expired"],
        "store credit": ["store credit", "store_credit"],
        "replacement": ["replacement", "replace"],
        "exchange": ["exchange", "swap"],
        "cancel": ["cancel", "cancellation"],
    }
    for action, keywords in actions.items():
        if any(k in exp_lower for k in keywords):
            if any(k in out_lower for k in keywords):
                score += 0.4
            break

    # Amount correctness (0.3)
    exp_amounts = re.findall(r'\$(\d+\.\d{2})', expected_resolution)
    if exp_amounts:
        out_amounts = re.findall(r'\$(\d+\.\d{2})', output_text or "")
        hits = sum(1 for a in exp_amounts if a in out_amounts)
        score += 0.3 * (hits / len(exp_amounts))
    else:
        score += 0.15

    # Policy reasoning (0.2)
    policy_terms = ["window", "restocking", "defective", "sale", "platinum", "gold",
                    "standard", "late", "shipping credit", "personal care", "eligible"]
    exp_terms = [t for t in policy_terms if t in exp_lower]
    if exp_terms:
        hits = sum(1 for t in exp_terms if t in out_lower)
        score += 0.2 * (hits / len(exp_terms))

    # Tool usage bonus (0.1)
    if output_tools:
        tool_names = [t.function.name if hasattr(t, 'function') else t.get("function", {}).get("name", "") for t in output_tools]
        if "get_order" in tool_names:
            score += 0.1

    return round(min(score, 1.0), 3)


### Load Validation Scenarios

Our validation set contains 57 carefully crafted scenarios covering all policy rules.

In [16]:
# Load validation scenarios
with open("data/rft_v7_val.jsonl") as f:
    val_scenarios = [json.loads(line) for line in f]

print(f"Loaded {len(val_scenarios)} validation scenarios")
print(f"Sample: {val_scenarios[0]['messages'][-1]['content'][:80]}...")


Loaded 57 validation scenarios
Sample: Hi, it’s Noah Brown. I need to return the hiking boots from order ORD-010 and ge...



### Generate Ground Truth Expected Resolutions

This section demonstrates how to generate high-quality expected resolutions using a trusted reference model.
We run every scenario through `gpt-5.4` and save the outputs as ground truth — a useful technique when you need to create or verify expected resolutions for new datasets.

- Results are cached to `data/rft_v7_val_gt.jsonl` — subsequent runs load instantly
- Uses `gpt-5.4` as the reference model (~15–20 min for all scenarios)


In [ ]:
import os

GT_FILE = "data/rft_v7_val_gt.jsonl"

if os.path.exists(GT_FILE):
    with open(GT_FILE) as f:
        val_scenarios = [json.loads(line) for line in f]
    print(f"✅ Loaded {len(val_scenarios)} scenarios with cached ground truth")
    sample = val_scenarios[0].get("expected_resolution", "")[:120]
    print(f"   Sample resolution: {sample}...")
else:
    print(f"Generating ground truth for {len(val_scenarios)} scenarios using gpt-5.4...")
    print("(~15–20 min — results cached to avoid re-running)\n")

    gt_scenarios = []
    for i, ex in enumerate(val_scenarios):
        msg = ex["messages"][-1]["content"]
        output, _ = run_agent(msg, model="gpt-5.4", verbose=False)
        ex_with_gt = dict(ex)
        ex_with_gt["expected_resolution"] = output
        gt_scenarios.append(ex_with_gt)
        print(f"  [{i+1:2d}/{len(val_scenarios)}] {msg[:70]}...")

    with open(GT_FILE, "w") as f:
        for ex in gt_scenarios:
            f.write(json.dumps(ex) + "\n")

    val_scenarios = gt_scenarios
    print(f"\n✅ Ground truth saved to {GT_FILE}")
    print(f"   Re-run this cell anytime to reload from cache")


Generating ground truth for 57 scenarios using gpt-5.4...
(~15–20 min — results cached to avoid re-running)



### Run the Baseline Evaluation

We sample 8 scenarios to keep the evaluation quick (~5–8 min).
Each scenario runs the full agent loop and scores the result.

In [ ]:
# Run baseline evaluation (takes ~5-8 minutes with 30 scenarios)
import random
random.seed(42)
eval_scenarios = random.sample(val_scenarios, min(8, len(val_scenarios)))

print(f"Evaluating base o4-mini on {len(eval_scenarios)} scenarios...\n")
base_scores = []

for i, ex in enumerate(eval_scenarios):
    msg = ex["messages"][-1]["content"]
    expected = ex.get("expected_resolution", "")

    output, tools = run_agent(msg, model="o4-mini", verbose=False)
    score = python_grader(output, tools, expected)
    base_scores.append(score)

    status = "✅" if score >= 0.9 else ("⚠️" if score >= 0.5 else "❌")
    print(f"  [{i+1:2d}] {score:.3f} {status}  {msg[:55]}")

base_avg = sum(base_scores) / len(base_scores)
base_p90 = sum(1 for s in base_scores if s >= 0.9) / len(base_scores)
base_p80 = sum(1 for s in base_scores if s >= 0.8) / len(base_scores)

print(f"\n{'='*50}")
print(f"  BASE o4-mini RESULTS")
print(f"  Average score: {base_avg:.1%}")
print(f"  Pass@0.9 (strict): {base_p90:.0%}")
print(f"  Pass@0.8 (good): {base_p80:.0%}")
print(f"{'='*50}")


Evaluating base o4-mini on 8 scenarios...

  [ 1] 0.900 ✅  Noah Brown, noah.brown@example.com. Can I exchange the 
  [ 2] 0.950 ✅  Noah Brown, noah.brown@example.com. I'd like to return 
  [ 3] 0.867 ⚠️  Ava Chen, ava.chen@example.com. My yoga mat from ORD-00
  [ 4] 0.850 ⚠️  Ava Chen, ava.chen@example.com. I want to return the he
  [ 5] 0.700 ⚠️  Emma Kim, emma.kim@example.com. The water bottle from O
  [ 6] 0.800 ⚠️  Yusuf Rossi, yusuf.rossi@example.com. Please cancel ord
  [ 7] 0.700 ⚠️  Sofia Martinez, sofia.martinez@example.com. I'd like to
  [ 8] 0.850 ⚠️  Yusuf Rossi, yusuf.rossi@example.com. Can I return the 

  BASE o4-mini RESULTS
  Average score: 82.7%
  Pass@0.9 (strict): 25%
  Pass@0.8 (good): 75%


---
## Section 4: The Grader — How RFT Learns (5 min)

RFT works differently from SFT (supervised fine-tuning):

| | SFT | RFT |
|---|-----|-----|
| Data | Prompt + ideal response pairs | Prompts only (+ grader) |
| Signal | "Copy this response" | "This attempt scored 0.85 — try to do better" |
| Best for | Teaching format/style | Improving reasoning/accuracy |

The **grader** is the key to RFT. It scores each training rollout, and the model learns to
maximize its score. Our grader gives **partial credit** — crucial for learning:

```
score = 0.4 × action_correct + 0.3 × amount_correct + 0.2 × policy_reasoning + 0.1 × tool_usage
```

The **pass_threshold** determines what score counts as success vs failure for the RL reward.
Let's see how different thresholds affect the training signal:

### Calibrate the Pass Threshold

The pass threshold determines what the RL algorithm considers "success" vs "failure".
We want a **25–50% failure rate** on the base model — this gives the strongest learning signal.

- Too easy (low failure rate) = model already passes most examples, nothing to learn from
- Too hard (high failure rate) = sparse reward signal, model can't find the right direction

In [ ]:
# Calibration: what threshold gives the right failure rate?
print("Pass threshold calibration (base o4-mini):\n")
print(f"  {'Threshold':>10} {'Pass Rate':>10} {'Fail Rate':>10} {'Signal Quality':>15}")
print(f"  {'-'*10} {'-'*10} {'-'*10} {'-'*15}")

for threshold in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    pass_rate = sum(1 for s in base_scores if s >= threshold) / len(base_scores)
    fail_rate = 1 - pass_rate
    quality = "✅ Good (25-50%)" if 0.25 <= fail_rate <= 0.50 else ("⚠️ Too easy" if fail_rate < 0.25 else "⚠️ Too hard")
    print(f"  {threshold:>10.2f} {pass_rate:>9.0%} {fail_rate:>9.0%} {quality:>15}")

print(f"\n💡 From our experiments pass_threshold=0.80 gives on average 35% failure rate → good learning signal")

Pass threshold calibration (base o4-mini):

   Threshold  Pass Rate  Fail Rate  Signal Quality
  ---------- ---------- ---------- ---------------
        0.50      100%        0%     ⚠️ Too easy
        0.60      100%        0%     ⚠️ Too easy
        0.70      100%        0%     ⚠️ Too easy
        0.80       75%       25% ✅ Good (25-50%)
        0.85       62%       38% ✅ Good (25-50%)
        0.90       25%       75%     ⚠️ Too hard
        0.95       12%       88%     ⚠️ Too hard

💡 We use pass_threshold=0.80 → ~35% failure rate → good learning signal


---
## 💡 Key Takeaways

- **Base o4-mini scores ~73%** on Zava policy scenarios — good but not great
- The **grader provides partial credit** (0.0–1.0) rather than binary pass/fail
- The **pass threshold of 0.80** gives a ~35% failure rate — optimal for RL learning
- RFT uses the grader to score model attempts and learn from trial and error

**Next → Open `04-build-data-submit-job.ipynb` to create training data and submit an RFT job.**